# Goal

Тестируем  `18d_world_model_05`, а именно:
1) факторизованную `WorldModel`: `ReconstructionModel` (spatial, structure) + `PredictionModel` (temporal, dynamics)
2) отчуждаемый `ReconstructionModel`
3) в `ReconstructionModel` используются раздельные `lv_encoding` (как это было в `18d_world_model_03`)
4) разные варианты `model.render_head.render_engine(type='block', ...)`
5) `with_prediction=False`
6) **БЕЗ `BatchNorm2d` в рендерере**

# GRID_SEARCH_SPACE

In [ ]:
# @launchit.collect
# GRID_SEARCH_SPACE = dict(
# )

# set_hyperparameters

In [25]:
# @launchit.collect
def set_hyperparameters(HP, optuna_study, optuna_trial):
    import random
    HP.general.comment = None
    HP.general.random_seed = random.randint(0, 100)
    HP.general.is_torch_deterministic = True
    HP.general.is_torch_compile = True
    HP.general.is_torch_amp = True
    
    HP.dataset.train = [
        'train_dataset:100',
        'train_dataset:101',
        'train_dataset:102',
        'train_dataset:103',
        'train_dataset:104',
        'train_dataset:105',
        'train_dataset:106',
        'train_dataset:107',
        'train_dataset:108',
        'train_dataset:109',
    ]
    HP.dataset.test = 'test_dataset:2'

    HP.model.parent = None
    HP.model.sequence_length = 4
    HP.model.actions_count = 6
    HP.model.ob_shape = (1, 178, 152)
    HP.model.d_model = 256
    HP.model.vision_head = dict(grid=(6,6), features_counts=(16, 32, 64, 128))
    HP.model.transformer = dict(layers_count=3, heads_count=4, attention_backend=['EFFICIENT_ATTENTION', 'MATH'])

    # Use strings to make optuna happy (otherwise optuna will complain that tuple is not a supported type for persistent storage)
    import ast
    block_layout = optuna_trial.suggest_categorical('block_layout', ['(3, 3)', '(4, 3)', '(5, 5)', '(6, 6)'])
    block_layout = ast.literal_eval(block_layout)
    HP.model.render_head = dict(
        projector=dict(type='linear'), 
        render_engine=dict(
            type='block', 
            layout=block_layout,
            is_batch_norm=False,
        )
    )
    
    HP.train.epochs_count = 100
    HP.train.batch_size = 128
    HP.train.optimizer = 'AdamW'
    HP.train.max_grad_norm = 1.0
    HP.train.learn_rate = 'const(0.0005)'
    HP.train.bce_loss_coef = 'const(1.0)'
    HP.train.edge_loss_coef = 'const(1.0)'
    HP.train.recon_loss_coef = 'const(1.0)'
    HP.train.pred_loss_coef = 'const(0)'
    HP.train.reg_loss_coefs = {}
    HP.train.with_prediction = False
    
    return HP
# @launchit.stop

# Results
<TBD>

Полный провал. Это даже хуже, чем `18d_study_07.1`. Лучший прогон дал `ssim=0.9699`, тогда как в `18d_study_07.1` было 0.9775.

Т.е. для layered рендерера `BatchNorm2d` противопоказан, я для блочного, наоборот показан! Тут интересное обсуждение состоялось: https://share.google/aimode/xZrX6TS7VMpuVMADM

<img src="./img/ssim.png">

**Выводы**
1) остаёмся с layered рендерером
2) прогнать чемпионов `18d_study_10.1` и `18d_study_08.1` повторно, чтобы выжать из них максимум!